# 6. Удержание участников по уровням
Как связаны уровень (Novice … Grandmaster), число соревнований и срок активности.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load, TIERS

sns.set_theme(style="whitegrid")
users = load("Users")
teams = load("Teams")[["Id", "CompetitionId", "ScoreFirstSubmittedDate"]]
members = load("TeamMemberships")[["TeamId", "UserId"]]

In [ ]:
part = members.merge(teams, left_on="TeamId", right_on="Id")
per_user = part.groupby("UserId").agg(
    n_comps=("CompetitionId", "nunique"),
    first=("ScoreFirstSubmittedDate", "min"),
    last=("ScoreFirstSubmittedDate", "max"),
)
per_user["active_days"] = (per_user["last"] - per_user["first"]).dt.days
per_user = per_user.join(users.set_index("Id")["PerformanceTier"].map(TIERS).rename("tier"))
per_user = per_user[per_user["tier"].notna() & (per_user["tier"] != "Staff")]

## Доля участников с одним соревнованием по уровням

In [ ]:
tier_order = [t for t in TIERS.values() if t != "Staff"]
one_comp = (per_user["n_comps"] == 1).groupby(per_user["tier"]).mean().reindex(tier_order)
one_comp

In [ ]:
one_comp.plot(kind="bar", figsize=(8, 4), title="Доля участников с одним соревнованием")
plt.savefig("../reports/figures/06_one_comp_by_tier.png", bbox_inches="tight")
plt.show()

## Срок активности и число соревнований

In [ ]:
per_user.groupby("tier")[["n_comps", "active_days"]].median().reindex(tier_order)

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(data=per_user, x="tier", y="active_days", order=tier_order, showfliers=False)
plt.title("Срок активности (дней) по уровням")
plt.show()

Уровень в Users — текущий, поэтому связь «уровень → удержание» частично обратная:
высокий уровень получают те, кто долго участвует.